In [ ]:
# %% [markdown]
# # Signed pairwise bend and twist distributions
#
# This notebook reads all files matching `*_pair.csv` in the current working
# directory. It is written for the current pairwise-analysis output format:
#
#     t,bend_signed_deg,twist_deg
#
# It produces four KDE distribution plots:
# 1. Signed bend distributions for every pair.
# 2. Twist distributions for every pair.
# 3. Distribution of the time-wise mean twist across all pairs.
# 4. Distribution of the time-wise mean signed bend across all pairs.

# %%
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# %%
# ---- Settings ----

# This must match the header produced by the current pairwise analysis script.
BEND_COLUMN = "bend_signed_deg"
TWIST_COLUMN = "twist_deg"

# Save PNG copies in the current working directory.
SAVE_FIGURES = True
N_GRID_POINTS = 600

# %%
# ---- Load pair files ----

pair_pattern = re.compile(r"^(\d+)_pair\.csv$")
pair_files = []

for path in Path.cwd().glob("*_pair.csv"):
    match = pair_pattern.match(path.name)
    if match:
        pair_files.append((int(match.group(1)), path))

pair_files.sort(key=lambda item: item[0])

if not pair_files:
    raise FileNotFoundError(
        "No files matching '*_pair.csv' were found in the current working directory."
    )

pair_data = {}
for pair_number, path in pair_files:
    df = pd.read_csv(path)
    required_columns = {"t", BEND_COLUMN, TWIST_COLUMN}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: {sorted(missing_columns)}. "
            f"Expected the current format: t,{BEND_COLUMN},{TWIST_COLUMN}"
        )

    for column in ("t", BEND_COLUMN, TWIST_COLUMN):
        df[column] = pd.to_numeric(df[column], errors="coerce")

    pair_data[str(pair_number)] = df

print("Loaded pair files:")
for label, df in pair_data.items():
    print(f"  {label}_pair.csv: {len(df)} frames")

# %%
# ---- Helper functions ----

def finite_values(values):
    """Return only finite numerical values."""
    values = np.asarray(values, dtype=float)
    return values[np.isfinite(values)]


def kde_curve(values, n_grid=N_GRID_POINTS):
    """Generate x/y data for a Gaussian KDE, including a constant-data fallback."""
    values = finite_values(values)
    if len(values) < 2:
        return None, None

    low, high = values.min(), values.max()
    if np.isclose(low, high):
        width = max(abs(low) * 0.01, 1e-3)
        x = np.linspace(low - 4 * width, high + 4 * width, n_grid)
        y = np.exp(-0.5 * ((x - low) / width) ** 2) / (width * np.sqrt(2 * np.pi))
        return x, y

    padding = 0.08 * (high - low)
    x = np.linspace(low - padding, high + padding, n_grid)
    try:
        return x, gaussian_kde(values)(x)
    except np.linalg.LinAlgError:
        return None, None


def plot_pair_distributions(data_by_pair, column, title, x_label, output_filename):
    """Draw one KDE curve for each monomer pair."""
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    colours = plt.cm.viridis(np.linspace(0.08, 0.92, len(data_by_pair)))
    plotted = 0

    for colour, (label, df) in zip(colours, data_by_pair.items()):
        x, y = kde_curve(df[column])
        if x is None:
            print(f"Warning: skipped pair {label}; insufficient valid {column} data.")
            continue
        ax.plot(x, y, lw=2, color=colour, label=f"Pair {label}")
        plotted += 1

    if plotted == 0:
        raise ValueError(f"No usable data were found for {column}.")

    ax.axvline(0, color="black", lw=0.9, alpha=0.45)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Probability density")
    ax.legend(title="Monomer pair", frameon=False)
    ax.grid(alpha=0.25)

    if SAVE_FIGURES:
        fig.savefig(output_filename, dpi=300, bbox_inches="tight")
    plt.show()


def timewise_mean(data_by_pair, column):
    """Align pair data by time t and calculate the mean over pairs at each time."""
    tables = []
    for label, df in data_by_pair.items():
        table = df[["t", column]].copy()
        table = table.rename(columns={column: f"pair_{label}"})
        tables.append(table)

    merged = tables[0]
    for table in tables[1:]:
        merged = merged.merge(table, on="t", how="outer")

    pair_columns = [column for column in merged.columns if column != "t"]
    merged["average"] = merged[pair_columns].mean(axis=1, skipna=True)
    return merged.sort_values("t")


def plot_mean_distribution(mean_df, title, x_label, output_filename):
    """Draw the KDE of the time-wise mean across all pairs."""
    x, y = kde_curve(mean_df["average"])
    if x is None:
        raise ValueError("Insufficient valid values for the average distribution.")

    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    ax.plot(x, y, color="#B2182B", lw=2.5, label="Mean across pairs")
    ax.fill_between(x, y, color="#B2182B", alpha=0.18)
    ax.axvline(0, color="black", lw=0.9, alpha=0.45)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Probability density")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

    if SAVE_FIGURES:
        fig.savefig(output_filename, dpi=300, bbox_inches="tight")
    plt.show()

# %% [markdown]
# ## 1. Signed-bend distributions for all pairs

# %%
plot_pair_distributions(
    pair_data,
    BEND_COLUMN,
    title="Pairwise signed-bend distributions",
    x_label="Signed bend (degrees; local positive direction)",
    output_filename="pairwise_signed_bend_distributions.png",
)

# %% [markdown]
# ## 2. Twist distributions for all pairs

# %%
plot_pair_distributions(
    pair_data,
    TWIST_COLUMN,
    title="Pairwise twist distributions",
    x_label="Twist (degrees; clockwise positive)",
    output_filename="pairwise_twist_distributions.png",
)

# %% [markdown]
# ## 3. Distribution of the time-wise mean twist

# %%
average_twist = timewise_mean(pair_data, TWIST_COLUMN)
plot_mean_distribution(
    average_twist,
    title="Distribution of average pairwise twist",
    x_label="Mean twist across pairs (degrees; clockwise positive)",
    output_filename="average_pairwise_twist_distribution.png",
)

# %% [markdown]
# ## 4. Distribution of the time-wise mean signed bend

# %%
average_bend = timewise_mean(pair_data, BEND_COLUMN)
plot_mean_distribution(
    average_bend,
    title="Distribution of average pairwise signed bend",
    x_label="Mean signed bend across pairs (degrees)",
    output_filename="average_pairwise_signed_bend_distribution.png",
)

# %%
# Optional: inspect the time-wise mean data used for the final two plots.
display(average_twist.head())
display(average_bend.head())
